# Notebook 0: Welcome & Setup

<img src="../assets/scigen_logo.png" width="150" align="right">

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

This notebook gets your Colab environment ready for the rest of the tutorial.
We will:
1. Verify GPU access
2. Clone the SCIGEN repository
3. Install all dependencies
4. Download pretrained model weights
5. Verify everything works

> **Run this notebook first**, then move on to the other notebooks in order.

---
## 0.1 Check GPU availability

Go to **Runtime > Change runtime type** and select **T4 GPU** before running this cell.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU available: {gpu_name}')
else:
    print('WARNING: No GPU detected!')
    print('Go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')
    print('The tutorial will still work on CPU, but generation will be much slower.')

---
## 0.2 Clone the SCIGEN repository

In [ ]:
import os, shutil

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

os.chdir('/content')

if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    !git clone {REPO_URL} {PROJECT_DIR}

assert os.path.isdir(os.path.join(PROJECT_DIR, 'scigen')), \
    f'Clone failed — scigen/ not found in {PROJECT_DIR}'

print(f'Repository ready: {PROJECT_DIR}')

---
## 0.3 Install dependencies

This installs PyTorch Geometric extensions (matching the Colab PyTorch/CUDA versions)
and the other libraries SCIGEN needs.

> **This takes ~2–5 minutes.** You can read ahead while it runs.

In [ ]:
import torch, os

# Detect PyTorch and CUDA versions for compatible wheel installation
_ver_parts = torch.__version__.split('+')[0].split('.')
TORCH_VER = f'{_ver_parts[0]}.{_ver_parts[1]}.0'
if torch.version.cuda:
    CUDA_VER = 'cu' + torch.version.cuda.replace('.', '')
else:
    CUDA_VER = 'cpu'
PYG_WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
os.environ['PYG_WHEEL_URL'] = PYG_WHEEL_URL
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda or "N/A (CPU)"}')
print(f'PyG wheel index: {PYG_WHEEL_URL}')

In [ ]:
%%time
# Install PyG extensions from prebuilt wheels
!pip install -q torch-scatter torch-sparse torch-cluster \
    -f $PYG_WHEEL_URL

# Install all tutorial dependencies from the shared requirements file
!pip install -q -r {PROJECT_DIR}/requirements-colab.txt

print('All dependencies installed.')

---
## 0.4 Configure environment variables

SCIGEN requires several environment variables to be set before importing its modules.

In [ ]:
import os, sys

PROJECT_DIR = '/content/APS_demo_SCIGEN'

os.environ['PROJECT_ROOT'] = PROJECT_DIR
os.environ['HYDRA_JOBS'] = PROJECT_DIR
os.environ['WANDB_DIR'] = os.path.join(PROJECT_DIR, 'wandb')
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'script'))

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Environment configured.')

---
## 0.5 Download pretrained model weights

Downloads the SCIGEN model checkpoint (~250 MB) from Figshare.
This is needed for Notebook 05 (generation).

In [ ]:
import json, zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve

MODEL_DIR = Path(PROJECT_DIR) / 'models'
MODEL_PATH = MODEL_DIR / 'mp_20'
FIGSHARE_API_URL = 'https://api.figshare.com/v2/articles/27778134'

need_download = not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt'))

if need_download:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    print('Querying Figshare API...')
    article = json.loads(urlopen(FIGSHARE_API_URL).read().decode())
    files = article.get('files', [])
    print(f'Found {len(files)} file(s)')

    for f in files:
        dest = MODEL_PATH / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        print(f'  Saved ({dest.stat().st_size / 1e6:.1f} MB)')
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()
else:
    print('Model already downloaded.')

# Verify
ckpts = list(MODEL_PATH.glob('*.ckpt'))
assert ckpts, f'No .ckpt files in {MODEL_PATH}. Download may have failed.'
print(f'Model ready: {[c.name for c in ckpts]}')

> **Troubleshooting:** If the download fails, try re-running the cell. If it fails repeatedly,
> you can manually download from [Figshare](https://doi.org/10.6084/m9.figshare.27778134)
> and upload the files to `models/mp_20/` in the Colab file browser.

import torch
import torch_geometric
import hydra
from importlib.metadata import version

checks = {
    'torch': torch.__version__,
    'torch_geometric': torch_geometric.__version__,
    'pymatgen': version('pymatgen'),
    'hydra': hydra.__version__,
}

# Optional packages
for pkg in ['chgnet', 'mp_api']:
    try:
        checks[pkg] = version(pkg)
    except Exception:
        checks[pkg] = '(not installed)'

for pkg, ver in checks.items():
    print(f'  {pkg:<20s} {ver}')

print(f'\n  CUDA available:      {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU:                 {torch.cuda.get_device_name(0)}')
print()
print('Setup complete! You are ready for the tutorial.')

In [ ]:
import torch
import torch_geometric
import hydra
from importlib.metadata import version

print(f'torch:           {torch.__version__}')
print(f'torch_geometric: {torch_geometric.__version__}')
print(f'pymatgen:        {version("pymatgen")}')
print(f'hydra:           {hydra.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
print()
print('Setup complete! You are ready for the tutorial.')

---
## References

- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y) | [GitHub](https://github.com/RyotaroOKabe/SCIGEN)
- **PyTorch:** Paszke et al., "PyTorch: An Imperative Style, High-Performance Deep Learning Library," *NeurIPS* (2019). [pytorch.org](https://pytorch.org)
- **PyTorch Geometric:** Fey & Lenssen, "Fast Graph Representation Learning with PyTorch Geometric," *ICLR Workshop* (2019). [GitHub](https://github.com/pyg-team/pytorch_geometric)
- **Hydra:** Yadan, "Hydra — A framework for elegantly configuring complex applications," (2019). [GitHub](https://github.com/facebookresearch/hydra)

---
## What's next?

Proceed to **Notebook 01: Crystal Structures** to learn how to work with crystal structures in Python.